In [1]:
# Cell 1 — clone the official Google Taskmaster-1 dataset
!git clone --depth 1 https://github.com/google-research-datasets/Taskmaster.git

import json, os

TM1_DIR = "Taskmaster/TM-1-2019"

with open(f"{TM1_DIR}/self-dialogs.json") as f:
    self_dialogs = json.load(f)

with open(f"{TM1_DIR}/woz-dialogs.json") as f:
    woz_dialogs = json.load(f)

print(f"Self-dialogs: {len(self_dialogs)}")
print(f"WOZ-dialogs: {len(woz_dialogs)}")
print(json.dumps(self_dialogs[0], indent=2)[:1500])  # inspect structure

fatal: destination path 'Taskmaster' already exists and is not an empty directory.
Self-dialogs: 7708
WOZ-dialogs: 5507
{
  "conversation_id": "dlg-00055f4e-4a46-48bf-8d99-4e477663eb23",
  "instruction_id": "restaurant-table-2",
  "utterances": [
    {
      "index": 0,
      "speaker": "USER",
      "text": "Hi, I'm looking to book a table for Korean fod."
    },
    {
      "index": 1,
      "speaker": "ASSISTANT",
      "text": "Ok, what area are you thinking about?"
    },
    {
      "index": 2,
      "speaker": "USER",
      "text": "Somewhere in Southern NYC, maybe the East Village?",
      "segments": [
        {
          "start_index": 13,
          "end_index": 49,
          "text": "Southern NYC, maybe the East Village",
          "annotations": [
            {
              "name": "restaurant_reservation.location.restaurant.accept"
            }
          ]
        },
        {
          "start_index": 13,
          "end_index": 25,
          "text": "Southern NYC",
     

In [3]:
# Cell 2 (complete, revised) — filter domains, heuristic labeling with confirm/cancel detection
import re
import pandas as pd

# Filter to pizza and coffee domains
TARGET_DOMAINS = ("pizza", "coffee")

def get_domain(dialog):
    return dialog.get("instruction_id", "").split("-")[0]

all_dialogs = [d for d in (self_dialogs + woz_dialogs) if get_domain(d) in TARGET_DOMAINS]
print(f"Filtered dialogs: {len(all_dialogs)}")

CHECKOUT_PATTERNS = re.compile(r"\b(that'?s all|checkout|place (my|the) order|finalize|go ahead and order|that will be all|ring it up|that completes)\b", re.I)
MENU_PATTERNS = re.compile(r"\b(what.*(have|offer|options|flavors)|menu|what kind of)\b", re.I)
GREETING_PATTERNS = re.compile(r"\b(hi|hello|hey there)\b", re.I)
CONFIRM_PATTERNS = re.compile(r"^\s*(yes|yeah|yep|yup|correct|that'?s (correct|right|it|exactly)|sounds good|perfect|exactly)\b", re.I)
CANCEL_PATTERNS = re.compile(r"\b(remove|cancel|scratch (that|the)|take .{0,15} off|don'?t want (that|the|it) anymore|change my mind)\b", re.I)
TOPPING_TEXT_PATTERNS = re.compile(r"\b(topped with|topping|extra cheese|whipped cream|syrup|on top|add (some|a bit of|extra))\b", re.I)

def label_utterance(text, slot_names):
    """Heuristic intent labeling using slot annotations + keyword rules."""
    text_l = text.lower()
    word_count = len(text_l.split())

    if any("topping" in s for s in slot_names):
        return "add_topping"
    if any(k in s for s in slot_names for k in ("drink", "pizza", "size", "crust", "milk")):
        return "order_item"
    if CANCEL_PATTERNS.search(text_l):
        return "cancel_item"
    if word_count <= 6 and CONFIRM_PATTERNS.search(text_l):
        return "confirm"
    if CHECKOUT_PATTERNS.search(text_l):
        return "checkout"
    if TOPPING_TEXT_PATTERNS.search(text_l):
        return "add_topping"
    if MENU_PATTERNS.search(text_l):
        return "ask_menu"
    if GREETING_PATTERNS.search(text_l):
        return "greeting"
    return "other"

# --- This loop is what actually builds df — it was missing from your last run ---
records = []
for dialog in all_dialogs:
    for utt in dialog.get("utterances", []):
        if utt.get("speaker") != "USER":
            continue
        text = utt.get("text", "").strip()
        if not text:
            continue
        slot_names = [
            ann.get("name", "").lower()
            for seg in utt.get("segments", [])
            for ann in seg.get("annotations", [])
        ]
        intent = label_utterance(text, slot_names)
        records.append({"text": text, "intent": intent})

print(f"Total labeled utterances: {len(records)}")

df = pd.DataFrame(records)
print(df["intent"].value_counts())

Filtered dialogs: 4580
Total labeled utterances: 47693
intent
other          22968
order_item     13190
confirm         5750
add_topping     3767
greeting        1146
checkout         501
ask_menu         363
cancel_item        8
Name: count, dtype: int64


In [4]:
# Cell 3 — load externally-generated synthetic dataset
import json
from google.colab import files
from sklearn.model_selection import train_test_split
import pandas as pd

with open("synthetic_data.json") as f:
    synthetic_raw = json.load(f)

with open("synthetic_confirm_cancel.json") as f:
    synthetic_extra = json.load(f)

synthetic_raw = synthetic_raw + synthetic_extra


ALLOWED_INTENTS = {"order_item", "add_topping", "ask_menu", "checkout", "confirm", "cancel_item", "greeting"}

syn_df = pd.DataFrame(synthetic_raw)
assert set(syn_df.columns) >= {"text", "intent"}, "JSON must have 'text' and 'intent' fields"

bad_intents = set(syn_df["intent"].unique()) - ALLOWED_INTENTS
if bad_intents:
    print(f"WARNING — unexpected intent labels found, dropping: {bad_intents}")
    syn_df = syn_df[syn_df["intent"].isin(ALLOWED_INTENTS)]

syn_df = syn_df.drop_duplicates(subset="text").reset_index(drop=True)
print(f"Loaded synthetic examples: {len(syn_df)}")
print(syn_df["intent"].value_counts())

# Stratified split (no template grouping available for externally-generated data,
# so we rely on the generation prompt's diversity requirement instead)
syn_train_df, syn_val_df = train_test_split(
    syn_df, test_size=0.15, stratify=syn_df["intent"], random_state=42
)
print(f"\nSynthetic train: {len(syn_train_df)}, synthetic val: {len(syn_val_df)}")

Loaded synthetic examples: 1433
intent
cancel_item    417
confirm        416
greeting       120
order_item     120
ask_menu       120
add_topping    120
checkout       120
Name: count, dtype: int64

Synthetic train: 1218, synthetic val: 215


In [5]:
# Cell 4 (revised) — merge with "other" downsampling
df_real = df.copy()

# Cap "other" — it's a heuristic fallback bucket, not a real intent signal,
# and at 60% of the data it teaches the model to default to it under uncertainty
MAX_OTHER = 3000
other_mask = df_real["intent"] == "other"
if other_mask.sum() > MAX_OTHER:
    other_sample = df_real[other_mask].sample(n=MAX_OTHER, random_state=42)
    df_real = pd.concat([df_real[~other_mask], other_sample], ignore_index=True)

real_train, real_val = train_test_split(
    df_real, test_size=0.15, stratify=df_real["intent"], random_state=42
)

train_df = pd.concat([real_train, syn_train_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
val_df = pd.concat([real_val, syn_val_df], ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Final train size: {len(train_df)}")
print(f"Final val size: {len(val_df)}")
print("\nTrain intent distribution:")
print(train_df["intent"].value_counts())

INTENT_LABELS = sorted(train_df["intent"].unique())
label2id = {label: i for i, label in enumerate(INTENT_LABELS)}
id2label = {i: label for label, i in label2id.items()}
print(f"\nLabels: {INTENT_LABELS}")

Final train size: 24784
Final val size: 4374

Train intent distribution:
intent
order_item     11313
confirm         5241
add_topping     3304
other           2550
greeting        1076
checkout         528
ask_menu         411
cancel_item      361
Name: count, dtype: int64

Labels: ['add_topping', 'ask_menu', 'cancel_item', 'checkout', 'confirm', 'greeting', 'order_item', 'other']


In [6]:
# Cell 4b — sanity check actual text, not just counts
pd.set_option("display.max_colwidth", None)
for intent in INTENT_LABELS:
    print(f"\n=== {intent} ===")
    print(train_df[train_df["intent"] == intent]["text"].sample(min(5, len(train_df[train_df["intent"] == intent]))).to_string(index=False))


=== add_topping ===
                     I'm picky so make one just pepperoni. As for the other, let's say pepperoni, sausage, onion, and green pepper.
                                                                                            I would like to add some whipped cream.
On one of them, I would like barbecue chicken barbecue. The other three, I would like pepperoni, mushroom, extra cheese, and bacon.
                            I think it comes with a whip, but could you have them add extra caramel drizzle and a shot of espresso?
                                                          Can you add whipped cream please with cinnamon a dash of cinnamon on top?

=== ask_menu ===
                      What kind they have?
                   What kind do they have?
                  What sizes do they have?
                What flavors do they have?
What options do i have with the iced ones?

=== cancel_item ===
   take off the small mushroom pizza
       forget the jalapenos 

In [8]:
# Cell 5 — install/import transformers, tokenize, build PyTorch Dataset class
!pip install -q transformers datasets scikit-learn

import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 64  # drive-thru utterances are short; 64 tokens is generous headroom

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IntentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )
        # squeeze(0) removes the batch dim tokenizer adds — Dataset returns single examples
        item = {key: val.squeeze(0) for key, val in encoding.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Convert string intent labels -> integer ids using the label2id map from Cell 4
train_dataset = IntentDataset(
    texts=train_df["text"],
    labels=train_df["intent"].map(label2id),
    tokenizer=tokenizer,
)
val_dataset = IntentDataset(
    texts=val_df["text"],
    labels=val_df["intent"].map(label2id),
    tokenizer=tokenizer,
)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Val dataset size: {len(val_dataset)}")

# Sanity check — inspect one tokenized example
sample = train_dataset[0]
print("\nSample keys:", sample.keys())
print("input_ids shape:", sample["input_ids"].shape)
print("Decoded text:", tokenizer.decode(sample["input_ids"], skip_special_tokens=True))
print("Label id:", sample["labels"].item(), "-> intent:", id2label[sample["labels"].item()])

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Train dataset size: 24784
Val dataset size: 4374

Sample keys: dict_keys(['input_ids', 'token_type_ids', 'attention_mask', 'labels'])
input_ids shape: torch.Size([64])
Decoded text: i want one meat lovers, one supreme, one onion pepperoni bacon and extra cheese, and one with extra pepperoni, extra sausage, and green peppers.
Label id: 6 -> intent: order_item


In [9]:
# Cell 6 — load DistilBERT, compute class weights, define weighted Trainer + metrics
import numpy as np
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, precision_recall_fscore_support, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_labels = len(INTENT_LABELS)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
).to(device)

# --- Class weights, to counter the order_item/other majority we saw in value_counts() ---
train_label_ids = train_df["intent"].map(label2id).values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(num_labels),
    y=train_label_ids,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)
print("\nClass weights (higher = rarer class, penalized more when misclassified):")
for label, w in zip(INTENT_LABELS, class_weights):
    print(f"  {label}: {w:.3f}")

# --- Custom Trainer that applies class weights in the loss ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# --- Metrics: macro-F1 is the headline metric (imbalance-robust), accuracy reported alongside ---
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro_f1 = f1_score(labels, preds, average="macro")
    weighted_f1 = f1_score(labels, preds, average="weighted")

    precision, recall, f1_per_class, support = precision_recall_fscore_support(
        labels, preds, average=None, labels=np.arange(num_labels), zero_division=0
    )

    metrics = {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
    }
    # Per-class F1 logged individually so you can see rare-class performance during training
    for i, label in enumerate(INTENT_LABELS):
        metrics[f"f1_{label}"] = f1_per_class[i]

    return metrics

Using device: cpu


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Class weights (higher = rarer class, penalized more when misclassified):
  add_topping: 0.938
  ask_menu: 7.538
  cancel_item: 8.582
  checkout: 5.867
  confirm: 0.591
  greeting: 2.879
  order_item: 0.274
  other: 1.215


In [15]:
OUTPUT_DIR = "./distilbert-drivethru-intent"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",  # avoids wandb prompt interrupting the run
    fp16=torch.cuda.is_available(),  # speeds up training on the T4 GPU
    optim="adamw_torch", # Use standard PyTorch AdamW to avoid fused optimizer XLA incompatibility
    remove_unused_columns=False, # Crucial for custom datasets to avoid label mismatch issues
    label_names=["labels"] # Explicitly tell Trainer which input key corresponds to the labels
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("\nTraining complete.")
print(train_result.metrics)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,F1 Add Topping,F1 Ask Menu,F1 Cancel Item,F1 Checkout,F1 Confirm,F1 Greeting,F1 Order Item,F1 Other
1,0.052113,0.319399,0.935327,0.932465,0.935162,0.885590,0.915493,0.977099,0.939227,0.980885,0.941772,0.939379,0.880274
2,0.041345,0.308073,0.937645,0.937832,0.937640,0.887139,0.913043,0.992248,0.949721,0.982094,0.963542,0.942623,0.872247
3,0.028418,0.329405,0.936949,0.938246,0.936736,0.883640,0.921986,0.984615,0.955556,0.982016,0.960836,0.940609,0.876712


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1,F1 Add Topping,F1 Ask Menu,F1 Cancel Item,F1 Checkout,F1 Confirm,F1 Greeting,F1 Order Item,F1 Other
1,0.052113,0.319399,0.935327,0.932465,0.935162,0.885590,0.915493,0.977099,0.939227,0.980885,0.941772,0.939379,0.880274
2,0.041345,0.308073,0.937645,0.937832,0.937640,0.887139,0.913043,0.992248,0.949721,0.982094,0.963542,0.942623,0.872247
3,0.028418,0.329405,0.936949,0.938246,0.936736,0.883640,0.921986,0.984615,0.955556,0.982016,0.960836,0.940609,0.876712
4,0.014826,0.352998,0.937181,0.938263,0.937067,0.885362,0.921986,0.992248,0.949721,0.981481,0.955614,0.941206,0.878484


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete.
{'train_runtime': 615.4037, 'train_samples_per_second': 161.091, 'train_steps_per_second': 10.068, 'total_flos': 1641711650930688.0, 'train_loss': 0.05219173600705998, 'epoch': 4.0}


In [16]:
# Cell 8 — final evaluation summary + per-class breakdown
eval_results = trainer.evaluate()
print("Final validation metrics:\n")
for k, v in eval_results.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training Loss,Validation Loss,Epoch,Accuracy,Macro F1,Weighted F1,F1 Add Topping,F1 Ask Menu,F1 Cancel Item,F1 Checkout,F1 Confirm,F1 Greeting,F1 Order Item,F1 Other
0.014826,0.352998,4,0.937181,0.938263,0.937067,0.885362,0.921986,0.992248,0.949721,0.981481,0.955614,0.941206,0.878484


Final validation metrics:

  eval_loss: 0.3530
  eval_accuracy: 0.9372
  eval_macro_f1: 0.9383
  eval_weighted_f1: 0.9371
  eval_f1_add_topping: 0.8854
  eval_f1_ask_menu: 0.9220
  eval_f1_cancel_item: 0.9922
  eval_f1_checkout: 0.9497
  eval_f1_confirm: 0.9815
  eval_f1_greeting: 0.9556
  eval_f1_order_item: 0.9412
  eval_f1_other: 0.8785


In [17]:
# Cell 9a — authenticate with Hugging Face Hub
from huggingface_hub import notebook_login
notebook_login()  # paste your HF write token when prompted

In [20]:
# Cell 9b — save locally and push to the Hub
REPO_NAME = "guth001/distilbert-drivethru-intent"  # change to your HF username

# Save tokenizer + model locally first
tokenizer.save_pretrained(OUTPUT_DIR)
trainer.save_model(OUTPUT_DIR)  # saves the BEST checkpoint, since load_best_model_at_end=True

# Also save the label mappings — your FastAPI backend will need these at inference time
import json
with open(f"{OUTPUT_DIR}/intent_labels.json", "w") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)

# Push everything to the Hub
# Move model to CPU before pushing to hub to avoid RuntimeError with safetensors
model.cpu()
model.push_to_hub(REPO_NAME)
tokenizer.push_to_hub(REPO_NAME)

# Push the label mapping file too, since push_to_hub() above only handles model/tokenizer files
from huggingface_hub import upload_file
upload_file(
    path_or_fileobj=f"{OUTPUT_DIR}/intent_labels.json",
    path_in_repo="intent_labels.json",
    repo_id=REPO_NAME,
)

print(f"\nModel pushed to: https://huggingface.co/{REPO_NAME}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...b1cl__n/model.safetensors:   0%|          |  575kB /  268MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]


Model pushed to: https://huggingface.co/guth001/distilbert-drivethru-intent
